**[🏠 Course Home](../README.md) | 🐍 Python Companion: [Appendix C: Verifying Flaky Test Fixes](../python/appendix_frequentist_vs_bayesian_flaky_tests.ipynb) | ↩️ Return to: [Chapter 8](08_case_studies_flaky_tests_and_pipeline_decisions.ipynb)**

---

# 🧪 Appendix C: Verifying Flaky Test Fixes — Frequentist vs. Bayesian Approaches
### *Why 0/100 Passes Proves Almost Nothing, and How Wald's SPRT, Bayes Factors, and Sequential Stopping Solve CI Flakiness*

---

## 1. What Are We Trying to Do?

In modern software engineering, intermittent test failures ("flaky tests") are a massive drain on developer velocity and company morale.
Every engineering team has lived through this exact dilemma:

```
Step 1: An integration test begins failing intermittently on main.
        In the last 100 runs, it failed twice:
        Pre-fix failure rate: 2 failures / 100 runs = 2.0%

Step 2: An engineer investigates, finds a likely race condition, and pushes a PR.

Step 3: To "prove" the fix works, they loop the test 100 times in CI:
        for i in {1..100}; do pytest test_billing.py; done

Step 4: All 100 runs pass without error (0 failures in 100 runs)!

Step 5: The engineer comments: "Fixed! Passed 100/100 runs."
        The PR is merged... and 48 hours later, it flakes in a production deployment!
```

Why does this happen so consistently?
In this dedicated conceptual appendix, we break down:
1. **The 50-Sided Die (Why 0/100 proves almost nothing)**.
2. **The Sugar Pill Trial (Fisher's Exact Test: $p = 0.25$)**.
3. **The Rule of Three ($p_{\text{upper}} \approx 3/N$)**.
4. **Wald's SPRT: The Balance Scale & Speed Camera**.
5. **The Courtroom Shoe Print (Bayes Factor $BF = 4.0$)**.
6. **The Fragile Porcelain Vase (Asymmetry of Evidence)**.
7. **The Hydraulic Shake Table (Stress Injection)**.

---

## 2. The 50-Sided Die (The 100-Pass Illusion)

Why did the test pass 100 times in a row if the bug was still present?

> [!IMPORTANT]
> **The Loaded Die Mental Model**
> 
> Imagine someone hands you a **50-sided die** where exactly **one face** is painted red (representing a 2% failure rate).
> * What is the probability that you roll this die **100 times** and the red face **never shows up at all**?
> * Each roll has a $49/50 = 98\%$ chance of landing safe.
> * The probability of 100 consecutive safe rolls is:
>   $$P(0 \text{ red in } 100) = (0.98)^{100} \approx \mathbf{13.3\%}$$
> 
> **Think about what 13.3% means in everyday life:**
> * Rolling a **6** on a standard 6-sided board game die has a probability of $1/6 \approx 16.7\%$.
> * A $13.3\%$ probability is roughly **1 out of every 7.5 times**!
> * If someone rolls a 6 on a single die roll, nobody claims the die is broken.
> * Yet software engineers celebrate 100 consecutive passes as "irrefutable proof" that a bug is permanently dead, when they have merely observed an event **almost as common as rolling a 6 on a board game die**!

---

## 3. Fisher's Exact Test & The Sugar Pill Clinical Trial

Let us analyze the engineer's experiment under classical frequentist hypothesis testing:

| Test Phase | Failures ($k$) | Passes ($n - k$) | Total Runs ($n$) | Empirical Rate ($\hat{p}$) |
| :--- | :---: | :---: | :---: | :---: |
| **Pre-Fix (Main Branch)** | 2 | 98 | 100 | 2.0% |
| **Post-Fix (PR Branch)** | 0 | 100 | 100 | 0.0% |

Fisher's Exact Test asks the clinical trial question:

> [!TIP]
> ### 💊 The Placebo Question
> Imagine a trial with 200 patients testing an anti-nausea medication against a sugar pill:
> * Placebo group: 2 patients vomit.
> * New drug group: 0 patients vomit.
> 
> If the drug is actually identical to the sugar pill, what are the odds that purely by the luck of the draw, both sick patients happened to be placed in the placebo group?
> 
> $$p = \frac{\binom{2}{2} \binom{198}{98}}{\binom{200}{100}} = \frac{100 \times 99}{200 \times 199} \approx \mathbf{0.2487} \quad (24.9\%)$$

**The Frequentist Verdict: Inconclusive!**
* There is a **25% chance** (1 out of 4!) of seeing this exact drop purely by random coin flips, even if the patch did **absolutely nothing**!
* The FDA would reject this drug instantly; CI systems should not accept it either!

---

## 4. The Rule of Three & Upper Confidence Bounds

When you observe zero events in $N$ trials, what is the frequentist 95% confidence upper bound on the true failure rate?
Under Clopper-Pearson exact binomial math, this simplifies to the famous **Rule of Three**:

$$p_{\text{upper}} \approx \frac{3}{N}$$

```
                   WHERE DOES THE "3" COME FROM?
                   
  If you expect 1 failure: P(0 fails) = e^(-1) ≈ 36.8% (Very common!)
  If you expect 2 failures: P(0 fails) = e^(-2) ≈ 13.5% (Still common!)
  If you expect 3 failures: P(0 fails) = e^(-3) ≈  5.0% (Finally drops below 5% doubt!)
  
  Conclusion: Zero failures only becomes surprising once you tested
  long enough that you EXPECTED to see 3 failures!
```

### The Paradox: You Proved Nothing!
* For $N = 100$ runs with 0 failures:
  $$p_{\text{upper}} \approx \frac{3}{100} = \mathbf{3.0\%}$$
* **Look at the numbers**: Before the fix, the test flaked at **2.0%**. After 100 clean passes, your guaranteed 95% upper bound is **3.0%**!
* **You have not proved the test is fixed; you have not even proved it is better than before!**

---

## 5. The Frequentist Solution: Wald's SPRT (The Tug-of-War Engine)

Running a fixed batch of 100 or 600 runs has two massive architectural flaws:
1. **If the fix FAILED** (e.g., the bug is still present and fails on run 3), finishing 97 more runs wastes cloud compute and delays developer feedback.
2. **If the fix SUCCEEDED**, we want to stop and merge the pull request the moment we clear the risk threshold, not wait for an arbitrary round number.

---

### 🎖️ The WWII Origin: Why Fixed Samples Waste Munitions and Compute

During World War II, statistician **Abraham Wald** worked at Columbia University's Statistical Research Group (SRG). 
Munitions testing was expensive and destructive: to test whether an artillery shell or bomb fuse was defective, you had to blow it up!

Traditional military protocol demanded firing a fixed batch of 500 shells before making a decision. Wald asked a radical question:
> *"Why fire 500 shells if the first 10 blow up in the barrel? And why fire 500 shells if the first 150 are flawless?"*

Wald invented the **Sequential Probability Ratio Test (SPRT)**, allowing inspectors to evaluate evidence after *every single observation*. The U.S. military classified SPRT as a military secret because it slashed inspection time and munitions consumption by **30% to 50%**.

In modern cloud CI/CD, your test runs are Wald's artillery shells: each run consumes AWS compute and developer time!

---

### 🪢 The Tug-of-War: The 3 Zones of SPRT

Think of Wald's SPRT as an automated tug-of-war between two competing hypotheses:
* **$H_0$ (Team Clean)**: The bug fix worked; failure rate is below acceptable baseline ($p_0 = 0.001$, or $0.1\%$).
* **$H_1$ (Team Flaky)**: The test is still broken; failure rate is at or above flaky rate ($p_1 = 0.02$, or $2.0\%$).

The system maintains a running evidence score $S_m$ starting at **$0.0$ (Neutral / Total Ignorance)**:

```text
                           THE SPRT PLAYING FIELD
                           
     +2.94 ─── RED LINE (Flaky Boundary A)   ──► STOP! Test is BROKEN! (Reject PR)
       ▲
       │
       │   [GRAY ZONE: Inconclusive / Not enough evidence yet / RUN ANOTHER TEST!]
       │
       │           ● Evidence Score starts at 0.0 (Neutral)
       │
       ▼
     -2.94 ─── GREEN LINE (Clean Boundary B) ──► STOP! Test is CLEAN! (Approve PR)
```

At any moment, one of three things is true:
1. **Score crosses above $+2.94$ (The Red Line)**: Team Flaky won. We have overwhelming statistical proof that the test is still broken. **Stop testing immediately and reject the PR.**
2. **Score crosses below $-2.94$ (The Green Line)**: Team Clean won. We have collected enough clean passes to prove the test is fixed. **Stop testing immediately and approve the PR.**
3. **Score is between $-2.94$ and $+2.94$ (The Gray Zone)**: The evidence is inconclusive. **Keep testing! Run test #2, test #3, etc.**

---

### 🔍 Where Does "2.94" Come From? The 19:1 Odds Derivation

The number **$2.94$** is not an arbitrary constant. It comes directly from standard **$95\%$ statistical certainty**:

1. **Set Your Error Tolerances ($\alpha$ and $\beta$)**:
   * $\alpha = 0.05$ (5% False Alarm rate): Chance of falsely rejecting a truly clean test.
   * $\beta = 0.05$ (5% Missed Defect rate): Chance of letting a broken flaky test slip into `main` ($95\%$ power).
2. **Calculate the Likelihood Ratio Thresholds**:
   Wald proved that the exact threshold for declaring a test flaky is:
   $$\text{Upper Threshold} = \frac{1 - \beta}{\alpha} = \frac{1 - 0.05}{0.05} = \frac{0.95}{0.05} = \mathbf{19}$$
   In betting odds, $19\text{ to }1$ means:
   $$\text{Probability} = \frac{19}{19 + 1} = \frac{19}{20} = \mathbf{95\%}$$
   To be $95\%$ confident, you must collect enough evidence until the data is **19 times more likely** under one hypothesis than the other!
3. **Take the Natural Logarithm ($\ln$)**:
   To prevent multiplying tiny decimals on a computer, Wald took the natural log, turning multiplication into simple addition:
   $$\text{Upper Boundary } A = \ln(19) \approx \mathbf{+2.944}$$
   $$\text{Lower Boundary } B = \ln\left(\frac{1}{19}\right) \approx \mathbf{-2.944}$$

---

### ⚖️ Sand vs. Sledgehammer: How Each Run Moves the Needle

After every test run, the computer adds or subtracts points based on the log-likelihood ratio:

1. **When a Test PASSES (A Tiny Grain of Sand)**:
   Even a broken flaky test passes $98\%$ of the time! Seeing a pass is not surprising; it provides only a tiny crumb of evidence:
   $$\Delta S_{\text{pass}} = \ln\left(\frac{1 - 0.02}{1 - 0.001}\right) = \ln\left(\frac{0.98}{0.999}\right) \approx \mathbf{-0.0192}\text{ points}$$
   It takes $\frac{-2.944}{-0.0192} \approx \mathbf{154\text{ consecutive passes}}$ of accumulating sand to push the needle down to the Green Line ($-2.94$) and prove the test is clean.
2. **When a Test FAILS (A Giant Sledgehammer)**:
   If the fix had worked, a failure should be virtually impossible ($0.1\%$). A failure post-fix is damning evidence:
   $$\Delta S_{\text{fail}} = \ln\left(\frac{0.02}{0.001}\right) = \ln(20) \approx \mathbf{+2.996}\text{ points}$$
   **A single failure drops a $+3.00$-point sledgehammer onto the scale!**
   Since the score started at $0.0$, one single failure instantly slams the score to $+2.996$, **blowing past the $+2.94$ Red Line on the spot**! The test loop aborts immediately on Run 1, 2, or 3.

---

### ⚡ What If We Can Run Tests in Parallel?

In real-world enterprise CI/CD (GitHub Actions, AWS CodeBuild, Kubernetes test clusters), you don't run tests one-by-one in serial. You might have **10, 20, or 50 worker pods running tests concurrently**.

How do Frequentist and Bayesian methods handle parallel batches?

```text
                     PARALLEL CI EXECUTION: 20 CONCURRENT PODS
                     
    Pod 1:  [PASS] ──► +1 Pass
    Pod 2:  [PASS] ──► +1 Pass
    Pod 3:  [FAIL] ──► EMERGENCY ABORT SIGNAL! (SIGTERM)
    Pod 4:  [RUNNING] ──► CANCELLED!
    ...
    Pod 20: [RUNNING] ──► CANCELLED!
```

#### 1. The Frequentist Parallel Solution: Batched SPRT with Cancellation Tokens
* **The Challenge**: Classical frequentist tests punish you for "peeking" at data early (inflating Type I false positive rates).
* **The Solution**:
  * Run tests in small parallel batches of size $B$ (e.g., 10 or 20 workers).
  * Update the score in chunks: $S_{\text{batch}} = S + \sum_{i=1}^B \Delta S_i$.
  * **The Cancellation Token**: Wire an asynchronous message broker (Redis/PubSub). The instant any worker records a **FAIL**, it broadcasts an immediate `SIGTERM` cancellation token to kill the remaining 19 workers on the spot!
  * If a whole batch passes, update the score by $B \times (-0.0192)$ and launch the next batch if still in the Gray Zone.

#### 2. The Bayesian Parallel Solution: Order-Invariance & The Async Pool
* **The Mathematical Miracle**: Under Bayes' rule, data likelihood is **exchangeable (order-invariant)**:
  $$P(y_1, y_2, \dots, y_B \mid \theta) = \prod_{i=1}^B P(y_i \mid \theta)$$
  Whether 20 tests arrive one-by-one over 20 minutes or finish simultaneously across 20 pods in 10 seconds, **the posterior distribution is 100% identical!**
  $$\alpha_{\text{new}} = \alpha_{\text{prior}} + k_{\text{failures}}, \qquad \beta_{\text{new}} = \beta_{\text{prior}} + (B - k_{\text{failures}})$$
* **Zero Peeking Penalty**: Bayesian inference has no concept of an "alpha-spending penalty." You can inspect the posterior after every single pod finishes, or after a batch of 50, without invalidating your statistics.
* **The Async Queue Engine**:
  Workers pull test runs from a shared job queue and write results atomically to a database row. An asynchronous evaluator checks after every job:
  $$\text{Is } P(\text{Flakiness} < 0.1\% \mid \text{Data}) \ge 95\%?$$
  The moment that threshold is cleared, the queue is drained, and the PR is stamped as verified!

---

## 6. The Bayesian Perspective: The Courtroom Shoe Print ($BF = 4.0$)

When an engineer loops a test 100 times locally and sees 100 passes, their gut screams:
> *"100 runs in a row without a single error! How could this possibly NOT be fixed?!"*

To see why human intuition fails so badly here, we must calculate the **Bayes Factor ($BF_{10}$)**.

The Bayes Factor asks a pure, unbiased comparison question:
> *"How much more likely is it to observe 100 clean passes if the fix actually worked, compared to if the fix was a total placebo and the bug is still alive?"*

$$BF_{10} = \frac{P(\text{100 passes} \mid M_1 \text{ [Fix Succeeded]})}{P(\text{100 passes} \mid M_0 \text{ [Fix Ineffective]})}$$

Let's plug in the actual probabilities:
1. **Under $M_1$ (Fix Succeeded)**: If the test has returned to a clean baseline ($<0.1\%$ failure rate), the probability of seeing 100 passes in a row is approximately **$50.0\%$** (integrating across clean baseline uncertainty).
2. **Under $M_0$ (Fix Ineffective / 2% Flaky)**: If the bug is still present and flaking at $2\%$, what is the probability of passing 100 runs by pure dumb luck?
   $$(0.98)^{100} \approx \mathbf{13.3\%} \quad (\text{or } 12.5\% \text{ under full Beta uncertainty})$$

Now, divide the two numbers:
$$BF_{10} = \frac{50.0\%}{12.5\%} \approx \mathbf{4.00}$$

---

### 👞 The Courtroom Shoe Print Drama

What does a Bayes Factor of **$4.0$** actually mean in real life?
Consider a classic criminal trial:

```text
                        THE COURTROOM FOOTPRINT TRIAL
                        
    The Crime Scene:       Burglary. Police find a size-10 boot print in the mud.
    The Suspect:           Arrested down the street wearing size-10 boots!
    
    The Prosecution:       "A MATCH! The evidence proves guilt beyond reasonable doubt!"
    The Defense:           "Wait: what percentage of adult men wear size-10 boots?"
                           --> Exactly 25% of men wear size 10!
```

Let's calculate the Bayes Factor of the footprint:
* If the suspect is **guilty**, he would leave a size-10 print ($100\%$ probability).
* If the suspect is an **innocent** man walking by, the chance he happens to wear size 10 is **$25\%$** (1 in 4 men).
* The Bayes Factor of the shoe print is:
  $$BF = \frac{100\%}{25\%} = \mathbf{4.0}$$

**Would any rational jury convict a person of burglary based solely on a size-10 shoe print?**
**Of course not!** A size-10 shoe print is *mild circumstantial evidence*. It eliminates people with size-7 and size-12 feet, but it leaves millions of innocent suspects on the table.

> [!IMPORTANT]
> ### 🔍 The Software Engineering Footprint
> **100 clean passes in CI is the exact software engineering equivalent of that size-10 shoe print!**
> It makes the fix *somewhat* more plausible, but it is nowhere near proof beyond reasonable doubt. It is weak circumstantial evidence disguised by the psychological illusion of the round number "100".

---

### 🧬 Prior Odds: Why Who Wrote the PR Matters

Frequentist statistics suffers from **structural amnesia**: it forces every PR to be evaluated as an identical black box, completely ignoring engineering reality.

In the real world, **what code changed in the PR matters immensely**. 
Using Bayes' rule in odds form:
$$\text{Posterior Odds} = \text{Prior Odds} \times BF_{10} \implies P(\text{Fixed} \mid \text{100 Passes}) = \frac{\text{Posterior Odds}}{1 + \text{Posterior Odds}}$$

Let's compare three common engineering archetypes when evaluated after 100 clean passes:

| Engineering PR Archetype | Prior Belief $P(\text{Fixed})$ | Prior Odds | Posterior Odds ($BF=4.0$) | Final Confidence After 100 Passes |
| :--- | :---: | :---: | :---: | :---: |
| **1. The `sleep()` Bump (Desperation Patch)**<br>Developer bumped `sleep(1)` to `sleep(2)`. Arbitrary sleep increases almost never solve underlying thread races. | **35%** | $0.35 / 0.65 = 0.54$ | $0.54 \times 4.0 = 2.15$ | **68.3%**<br>*(Nearly 1-in-3 chance the bug will still flake in production!)* |
| **2. The Standard Team PR**<br>Average refactor. Historical team telemetry shows ~70% of intended flaky fixes actually work. | **70%** | $0.70 / 0.30 = 2.33$ | $2.33 \times 4.0 = 9.33$ | **90.3%**<br>*(Bought only a 20% boost in confidence; 10% risk remains!)* |
| **3. The Deterministic Root-Cause Fix**<br>Developer reproduced the race condition, found the missing mutex lock on a shared dictionary, and proved the fix in unit tests. | **95%** | $0.95 / 0.05 = 19.0$ | $19.0 \times 4.0 = 76.0$ | **98.7%**<br>*(Certified clean; ready for production merge!)* |

#### The Bayesian Insight:
100 clean passes is **not enough to rescue a sloppy fix** (Archetype 1 stays at a dangerous $68\%$).
To verify an attempted fix with real statistical confidence, you must either:
1. Accumulate **~154 passes** sequentially via Wald's SPRT, or
2. Combine code review priors with **Stress Injection** (Section 8).

---

### 📐 The Bayes Factor Master Scale: What Numbers Should You Look For?

When evaluating statistical evidence—whether in code reviews, CI test suites, A/B experiments, or scientific research—what value of $BF_{10}$ should actually convince us to take action?

In Bayesian inference, two landmark frameworks are universally used to grade evidence: **Harold Jeffreys (1961)** and **Kass & Raftery (1995)**.

| Bayes Factor ($BF_{10}$) | Evidence Tier (Jeffreys / Kass & Raftery) | Real-World Translation | If Prior Was 50/50, Posterior Is... | Recommended Engineering Action |
| :--- | :--- | :--- | :---: | :--- |
| **$1$** | **No Evidence (Neutral)** | Absolute toss-up; the data taught us nothing. | **50.0%** | Continue testing or redesign the benchmark. |
| **$1 \text{ to } 3$** | **Anecdotal / "Barely worth mentioning"** | Background static, rumors, gut feel. | **50% – 75%** | **Do NOT merge.** Easily generated by random noise. |
| **$3 \text{ to } 10$** | **Substantial / Moderate** | Clear signal, but alternative explanations remain. *(Our 100 CI passes sits here at $BF = 4.0$)* | **75% – 91%** | Mild circumstantial evidence. Merge only if code review prior is already high ($>70\%$). |
| **$10 \text{ to } 30$** | **Strong** | Firm evidence; shifts ordinary decisions decisively. *(Wald's SPRT boundary $A = 19$ sits here)* | **91% – 97%** | **Safe CI automated merge gate.** Overcomes moderate developer skepticism. |
| **$30 \text{ to } 100$** | **Very Strong** | Robust, airtight proof. | **97% – 99%** | Standard for core architectural refactors or scientific discovery. |
| **$> 100$** | **Decisive / Overwhelming** | Open-and-shut case; unquestionable. *(Stress injection achieves $BF > 200$ in 15 runs)* | **> 99.0%** | Production-certified for mission-critical, zero-tolerance infrastructure. |

---

### ⚠️ The $p$-Value Trap: Why $p = 0.05$ Is Often Weak Evidence

A common mistake among engineers and researchers is assuming that a "statistically significant" test ($p < 0.05$) represents strong proof.

In a landmark paper, statisticians **Sellke, Bayarri, and Berger (2001)** calculated the maximum possible Bayes Factor for any given $p$-value:
> **A frequentist $p$-value of $0.05$ corresponds to a Bayes Factor between only $2.5$ and $3.5$!**

Look at the Master Scale above: a $BF$ of $2.5$ to $3.5$ falls squarely in the **"Anecdotal / Barely Worth Mentioning"** or bottom of "Moderate" tier!
* Classical hypothesis testing often flags a "statistically significant discovery" when the data only nudges the odds by $2.5\times$ or $3\times$.
* In CI/CD, if you rely on a naive significance threshold of $p < 0.05$, you will routinely certify flaky tests as "fixed" when they have barely escaped the level of background noise.

---

### 🪒 The Built-in Occam's Razor: Why Bayes Factor Penalizes Bloat

Why doesn't a complex, multi-parameter model always win a Bayes Factor contest against a lean model?

Because the Bayes Factor evaluates the **marginal likelihood (model evidence)**, which integrates performance over all possible parameter values under the prior:

$$P(\text{Data} \mid M) = \int P(\text{Data} \mid \theta, M) \, P(\theta \mid M) \, d\theta$$

```text
       MODEL COMPLEXITY VS. PROBABILITY DENSITY
       
  Probability
    Density
      ▲
      │       Simple Model (Sharp prediction)
      │          ┌───┐
      │         /     \
      │        /   ●   \  <-- Data lands in the bullseye!
      │       /         \     High marginal likelihood.
      │      /           \
      │   ──┴─────────────┴─────────────────────────
      │       Complex Model (Vague, 20 parameters)
      │     ───────────────────●───────────────────   <-- Spread paper-thin!
      │                                                   Low marginal likelihood.
      └─────────────────────────────────────────────► Possible Outcomes
```

* **The Lean Model ($M_{\text{lean}}$)**: Makes precise, focused predictions. If the observed data lands in that bullseye, its marginal likelihood density is high.
* **The Overfitted Model ($M_{\text{bloated}}$)**: Can explain almost anything by tweaking its 20 parameters, but because its probability mass is spread thin across a massive space of possibilities, its density at the actual data point is tiny.

> **Key Takeaway**: Unlike maximum likelihood, which always favors more parameters, **the Bayes Factor penalizes model bloat automatically** without needing ad-hoc penalties like AIC or BIC!

---

### 🎯 Contextual Decision Rules: Matching $BF$ to the Cost of Failure

How high of a Bayes Factor should you demand before declaring a problem solved?
It depends entirely on the **asymmetry of consequences**:

```text
               DECISION CONTEXT VS. TARGET BAYES FACTOR
               
   Low Stakes / Reversible                 High Stakes / Production Critical
  (e.g., UI tweak, local dev)             (e.g., DB migration, medical, safety)
  
          BF ≈ 3 - 6                             BF ≥ 20 - 100+
  ┌─────────────────────────┐             ┌─────────────────────────┐
  │ Moderate signal enough  │             │ Requires decisive proof │
  │ to pick a direction.    │             │ before pulling trigger. │
  └─────────────────────────┘             └─────────────────────────┘
```

1. **Low Stakes / Reversible Decisions (Dev / Sandbox)**:
   * **Target: $BF \ge 3 \text{ to } 6$**
   * If a false alarm costs nothing and a rollback takes 2 minutes, don't waste hours running thousands of tests.
2. **Standard Automated CI Gating (Team PRs)**:
   * **Target: $BF \ge 19.0$** (Wald's SPRT boundary for $\alpha=0.05, \beta=0.05$).
   * Pushes confidence to $>95\%$, preventing broken builds from blocking colleagues.
3. **Mission-Critical / Core Infrastructure (Database Migrations, Payment Services)**:
   * **Target: $BF \ge 100+$** (Decisive tier).
   * A broken migration can corrupt production databases or cause company-wide outages. Combine high run counts with **Stress Injection** (Section 8) to clear $BF > 200$.

---

## 7. The Asymmetry of Evidence: The Fragile Porcelain Vase

Why does verifying flaky test fixes feel so frustrating to developers?
Because in probability, **evidence is violently asymmetric**:

```text
                         THE ASYMMETRY OF EVIDENCE
                         
      Building Trust: 100 Clean Passes    ──►  Confidence rises slowly:  70% ──► 90.3%
      (Carrying the porcelain vase)
      
      Destroying Trust: 1 Single Failure   ──►  Confidence COLLAPSES:      70% ──► 36.8%!
      (Dropping the vase on the floor)
```

* **Building Trust Is Slow (Carrying the Porcelain Vase)**:
  To build trust, you must carry a fragile vase across 150 consecutive steps without a single slip. Each clean pass is expected, so it nudges your confidence upward by only a tiny fraction of a percentage point.
* **Destroying Trust Is Instant (Dropping the Vase)**:
  If a test fails just **once** (e.g., on run #8 post-fix), the illusion evaporates instantly.
  Under Bayes' rule:
  $$BF_{\text{fail}} = \frac{P(\text{Fail} \mid \text{Clean})}{P(\text{Fail} \mid \text{Broken})} = \frac{0.001}{0.02} = \mathbf{\frac{1}{20}} = \mathbf{0.05}$$
  A single failure slashes your posterior odds by **$20\times$**, plunging belief from $70\%$ down to **$36.8\%$** in a single second!

This asymmetry is why naive heuristics fail: **passes only whisper, but failures scream.**

---

## 8. The Secret Weapon: Stress Injection (The Hydraulic Shake Table)

If running 600 idle tests is too slow and expensive, what is the ultimate engineering solution? **Physics of Failure**.

> [!IMPORTANT]
> ### 🏗️ The Hydraulic Shake Table
> Civil engineers do not wait 50 years for an earthquake to test a skyscraper; they build a scale model and vibrate it violently on a **hydraulic shake table**.
> 
> In software engineering:
> * Do not run an idle test 600 times waiting for a 1% race condition to happen naturally.
> * **Inject Contention**: Throttle CPU to 1 core, inject 150ms synthetic network latency, spin up 16 concurrent background threads.
> * If stress injection inflates the failure rate from **$2\% \to 30\%$**:
>   * The chance of a broken test passing 15 runs drops to $(1 - 0.30)^{15} \approx \mathbf{0.47\%}$!
>   * Under stress, **just 15 consecutive passes provides a Bayes Factor $> 200$ (decisive mathematical proof)** while slashing CI execution time by **$95\%$**!

---

**[🏠 Course Home](../README.md) | 🐍 Python Companion: [Appendix C: Verifying Flaky Test Fixes](../python/appendix_frequentist_vs_bayesian_flaky_tests.ipynb) | ↩️ Return to: [Chapter 8](08_case_studies_flaky_tests_and_pipeline_decisions.ipynb)**
